# Exercise: Exploring decision trees with dtreeviz *(solution)*

- Run: [**Open In Colab**](https://colab.research.google.com/github/HassanAlgoz/AAI/blob/main/courses/Machine_Learning/exercises/11/dtree_viz_solution.ipynb)
- Dataset: [Titanic passenger records](https://raw.githubusercontent.com/parrt/dtreeviz/master/data/titanic/titanic.csv) (via the [dtreeviz](https://github.com/parrt/dtreeviz) repo)

---

Your model reports **82% accuracy** on Titanic survival. A stakeholder leans over your shoulder and asks:

> *"Great — but **why** did the model say this passenger would perish?"*

Accuracy alone cannot answer that. In this exercise you will explore [**dtreeviz**](https://github.com/parrt/dtreeviz) — a library that turns a fitted decision tree into rich, interpretable visuals.

**Learning objectives:**

- Wrap a scikit-learn tree in a dtreeviz **adaptor** with `dtreeviz.model()`
- Explore four areas of the API:
  1. **Tree structure** — see the full decision logic
  2. **Prediction paths** — trace one passenger through the tree
  3. **Leaf information** — inspect where training samples end up
  4. **Feature-space partitioning** — see how splits carve up feature space

**How each round works:**

1. **I do** — we run one worked example together
2. **The menu** — we list related functions/parameters (hints, not answers)
3. **You do** — you complete a task using only what was introduced so far

Pause before each *You do* cell and write down what you **expect** to see, then compare with the output.

## Setup

In [ ]:
# dtreeviz renders trees as SVG via the graphviz `dot` binary.
# On Colab this usually works out of the box; locally you may need:
#   sudo apt install graphviz   (Linux)
#   brew install graphviz       (macOS)
%pip install -q dtreeviz

In [ ]:
import pandas as pd

# https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
from sklearn.tree import DecisionTreeClassifier

# https://github.com/parrt/dtreeviz
import dtreeviz

RANDOM_STATE = 1234  # reproducible trees

### Load and prepare the Titanic data

We predict whether a passenger **survived** using six numeric features. Categorical columns are label-encoded so the tree can split on them.

In [ ]:
dataset_url = (
    "https://raw.githubusercontent.com/parrt/dtreeviz/master/data/titanic/titanic.csv"
)
dataset = pd.read_csv(dataset_url)

# Impute missing ages with the column mean (same as the dtreeviz examples)
dataset.fillna({"Age": dataset["Age"].mean()}, inplace=True)

# Label-encode categoricals for tree splitting
dataset["Sex_label"] = dataset["Sex"].astype("category").cat.codes
dataset["Cabin_label"] = dataset["Cabin"].astype("category").cat.codes
dataset["Embarked_label"] = dataset["Embarked"].astype("category").cat.codes

features = [
    "Pclass",
    "Age",
    "Fare",
    "Sex_label",
    "Cabin_label",
    "Embarked_label",
]
target = "Survived"

dataset[[target] + features].head()

### Train a shallow classifier and build the dtreeviz adaptor

The adaptor wraps your fitted scikit-learn model so every dtreeviz function shares the same interface.

In [ ]:
tree_classifier = DecisionTreeClassifier(
    max_depth=3,
    random_state=RANDOM_STATE,
)
tree_classifier.fit(dataset[features].values, dataset[target].values)

viz_model = dtreeviz.model(
    tree_classifier,
    X_train=dataset[features],
    y_train=dataset[target],
    feature_names=features,
    target_name=target,
    class_names=["perish", "survive"],
)

print(f"Tree trained on {len(dataset)} passengers, max_depth=3")

---

## Round 1 — Tree structure views

### Step 1 — I do: the default tree diagram

Run the cell below. Each node shows the split question, how many training samples reached it, and the class distribution at that node.

In [ ]:
viz_model.view(
    scale=0.8,  # shrink the SVG so it fits in the notebook
)

### Step 2 — The menu: other ways to call `view()`

The adaptor method `view()` accepts several parameters that change what you see:

| Parameter | What it does |
|-----------|-------------|
| `orientation="LR"` | Draws the tree left-to-right instead of top-down |
| `fancy=False` | Simpler nodes — useful when the tree is large |
| `depth_range_to_display=(start, end)` | Shows only nodes between two depth levels (root = level 0) |
| `scale` | Shrinks or enlarges the entire SVG |

All of these are passed to the **same** `viz_model.view(...)` call — you combine them as needed.

### Step 3 — You do: a compact overview for a presentation slide

Your manager wants a **single slide** that shows how the tree makes its first few decisions — nothing deeper than level 2, laid out **left-to-right**, in the **simple** (non-fancy) style.

Before you code, predict:

- How many split nodes will appear if you stop at depth 2?
- Will the slide be wider or taller with left-to-right orientation?

Use `viz_model.view(...)` with the right combination of parameters from the menu above.

In [ ]:
viz_model.view(
    orientation="LR",              # left-to-right layout for a wide slide
    fancy=False,                   # simpler nodes — less visual clutter
    depth_range_to_display=(0, 2), # root (0) through depth 2 only
)

---

## Round 2 — Prediction path explanations

This is where we answer the stakeholder's *"why this passenger?"* question.

### Step 1 — I do: highlight one passenger's route through the tree

In [ ]:
passenger_idx = 10
x = dataset[features].iloc[passenger_idx]

print(f"Passenger at row {passenger_idx}:")
display(x.to_frame().T)

print(f"Actual survival label: {dataset[target].iloc[passenger_idx]}")
print(f"Model prediction:      {tree_classifier.predict(x.values.reshape(1, -1))[0]}")

In [ ]:
viz_model.view(
    x=x,  # highlight this passenger's path through the tree
)

### Step 2 — The menu: dig deeper into one prediction

| Method / parameter | What it does |
|--------------------|-------------|
| `view(x=..., show_just_path=True)` | Shows **only** the nodes on this passenger's path — no siblings |
| `explain_prediction_path(x)` | Returns a **text** walk-through of every split comparison |
| `instance_feature_importance(x, figsize=...)` | Bar chart of feature importances **for this one prediction** |

These all take the same feature vector `x` you built above.

### Step 3 — You do: explain one passenger end-to-end

Pick **any** passenger index (try a few!), store their features in `x`, then:

1. Visualise **only** their decision path through the tree
2. **Print** the textual explanation of every split they encountered
3. Plot which features mattered most **for that single prediction**

Before you run all three, write down which feature you think will dominate — then check the bar chart.

In [ ]:
passenger_idx = 42
x = dataset[features].iloc[passenger_idx]

# 1. Only the nodes on this passenger's path
viz_model.view(
    x=x,
    show_just_path=True,
)

# 2. Textual walk-through of every split
print(viz_model.explain_prediction_path(x))

# 3. Per-instance feature importance bar chart
viz_model.instance_feature_importance(
    x,
    figsize=(3.5, 2),
)

---

## Round 3 — Leaf information

Leaves are where predictions are made. Understanding them tells you **where the model is confident** and **where it is guessing**.

### Step 1 — I do: how many samples land in each leaf?

In [ ]:
viz_model.leaf_sizes(
    figsize=(3.5, 2),  # width, height in inches
)

### Step 2 — The menu: other leaf-level diagnostics

| Method | What it does |
|--------|-------------|
| `ctree_leaf_distributions(figsize=...)` | Stacked bar chart of class counts **per leaf** |
| `leaf_purity(figsize=...)` | How "pure" each leaf is (one class dominates = high purity) |
| `node_stats(node_id=...)` | Detailed statistics for **one specific node** (splits and leaves both have IDs) |

Node IDs appear in the tree diagram from Round 1. The root is typically node 0.

### Step 3 — You do: find the weakest leaf and inspect it

A responsible ML workflow checks whether the model relies on **small** or **impure** leaves (signs of overfitting).

1. Plot **class distributions** and **leaf purity** side by side (two separate calls)
2. From those plots, identify the leaf that is either the **smallest** or the **least pure** (most mixed classes)
3. Call `node_stats(node_id=...)` with that leaf's ID and read the output

Before you pick a node, predict: will the least-pure leaf also be the smallest?

In [ ]:
viz_model.ctree_leaf_distributions(
    figsize=(3.5, 2),
)

viz_model.leaf_purity(
    figsize=(3.5, 2),
)

# Node 6 is a leaf in the max_depth=3 Titanic tree (see Round 1 diagram).
# Your answer may differ if you pick a different leaf from the plots.
leaf_node_id = 6

viz_model.node_stats(
    node_id=leaf_node_id,
)

---

## Round 4 — Feature-space partitioning

Decision trees partition feature space into regions. dtreeviz lets you **see** those regions — invaluable for understanding what the tree learned.

### Step 1 — I do: how does the tree split `Age`?

In [ ]:
viz_model.ctree_feature_space(
    features=["Age"],
    show={"splits", "title"},  # draw vertical split lines + a title
    figsize=(5, 1.5),
)

### Step 2 — The menu: customise feature-space plots

| Parameter | What it does |
|-----------|-------------|
| `nbins` | Number of histogram bins along each axis |
| `gtype='barstacked'` | Stacked-bar style instead of the default |
| `features=[feat_a, feat_b]` | Two features → **2D** partition view |
| `figsize` | Width and height of the figure |
| `show` | Set of annotations to include, e.g. `{'splits', 'title'}` |

For classification trees the method is `ctree_feature_space`; regression trees use `rtree_feature_space` instead.

### Step 3 — You do: two features, one picture

Ticket price (`Fare`) and age often interact on Titanic — wealthy adults had very different survival odds than children in steerage.

Visualise how your tree partitions the **2D** feature space spanned by `Age` and `Fare`. Include split lines and a title.

Before you run it, predict:

- Will the regions be axis-aligned rectangles? (Hint: decision trees always split on one feature at a time.)
- Which feature do you expect to be split on first — `Age` or `Fare`?

In [ ]:
viz_model.ctree_feature_space(
    features=["Age", "Fare"],      # two features → 2D partition
    show={"splits", "title"},      # vertical/horizontal split lines + title
    figsize=(5, 4),                # taller figure for 2D view
)

---

## Wrap-up — Reflect

Answer these in a markdown cell or your notes:

1. **Stakeholder communication:** Which of the four dtreeviz areas would you show a non-technical manager first — and why?
2. **Debugging:** You notice a leaf with only 3 training samples and 60/40 class mix. What would you try next on the *model* side (think back to lesson 11: `max_depth`, `min_samples_leaf`, …)?
3. **Connecting the dots:** How does the feature-space plot from Round 4 relate to the tree diagram from Round 1? Are they showing the same information differently?
4. **Going further:** The [dtreeviz repo](https://github.com/parrt/dtreeviz) also supports regression trees (`rtree_feature_space`), gradient boosting models (XGBoost, LightGBM), and even an AI `chat()` mode. Which would be most useful for your next project?

**Key takeaway:** Accuracy tells you *how often* the model is right; dtreeviz helps you understand *why* — a skill that outlasts any single dataset or library version.